# Example 5.2 / 1: Choosing a Value

Companion notebook for **5.2 Conditional and Logical Expressions** in the AMPL lab Documentation, section "Choosing a Value". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Choosing a Value

The model computes unmet demand and a review flag. The first expression returns a number; the
second returns `1` or `0`, which is convenient for reporting a logical condition.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # Products
param demand {PROD} >= 0;         # Units demanded
param capacity {PROD} >= 0;       # Units that can be supplied
param critical {PROD} binary;     # 1 if the product is critical

param shortage {p in PROD} =      # Unmet demand, 0 if capacity suffices
    if demand[p] > capacity[p] then demand[p] - capacity[p] else 0;

param needs_review {p in PROD} =  # 1 for a critical product that is short
    if critical[p] = 1 and shortage[p] > 0 then 1 else 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["demand"] = {"bands": 80, "coils": 120, "plate": 70}
ampl.param["capacity"] = {"bands": 100, "coils": 90, "plate": 60}
ampl.param["critical"] = {"bands": 1, "coils": 1, "plate": 0}

In [5]:
# run.py
shortages = ampl.param["shortage"].to_dict()
reviews = ampl.param["needs_review"].to_dict()
for product in ampl.set["PROD"].to_list():
    shortage = int(shortages[product])
    review = int(reviews[product])
    print(f"{product}: shortage={shortage}, review={review}")

bands: shortage=0, review=0
coils: shortage=30, review=1
plate: shortage=10, review=0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param: demand capacity critical :=
    bands   80  100  1
    coils  120   90  1
    plate   70   60  0;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display shortage, needs_review;

:     shortage needs_review    :=
bands      0         0
coils     30         1
plate     10         0
;

